# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdrayan001/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Connect to the FlyRank warehouse securely

import os
import duckdb

HF_TOKEN = os.getenv("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN is not available."

con = duckdb.connect()

con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])

con.execute("""
    CREATE OR REPLACE SECRET hf
    (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

MAR = f"{FACT}/month=2026-03/*.parquet"
DIM = f"{REL}/dim_content.parquet"
CLI = f"{REL}/dim_clients.parquet"

print("Connected to FlyRank warehouse.")
print("Month selected: 2026-03")

Connected to FlyRank warehouse.
Month selected: 2026-03


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

The warehouse grain is **one content page for one client on one report date**.

The main time field is `report_date`. For this assignment, I will develop on the **March 2026** monthly partition (`month=2026-03`), which is a mid-panel month rather than the final sealed month.

The page-level decision features will be built from information observed within this March 2026 window. The warehouse history is not balanced across clients, so availability flags must be checked before using engagement fields.

In [2]:
from pathlib import Path
import pandas as pd

# Find the starter dataset
paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv")
]

data_path = next(p for p in paths if p.exists())

# Load the dataset
df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Dataset loaded successfully.
Rows: 30000
Columns: 44


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify the unit of analysis and the available time-window fields

grain_check = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM read_parquet('{MAR}')
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").fetchdf()

print("Duplicate grain rows:")
print(grain_check)

assert grain_check.empty

print("\nGrain check passed.")
print("One row = one client × content × report date.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows:
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, row_count]
Index: []

Grain check passed.
One row = one client × content × report date.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field contract

**Features**
- `impressions_90d` — shows the amount of search visibility a page received.
- `sessions_90d` — shows the amount of user engagement from the page.
- `ctr` — shows how often search impressions resulted in clicks.
- `avg_position` — shows the page's average search ranking position.
- `content_age_days` — shows how old the content is.

**Label / proxy**
- `trend_direction` — used as a proxy for whether a page is showing a current decline signal. It is not a future outcome.

**Context**
- `content_id` — identifies the content page.
- `client_id` — identifies the client grouping used for analysis and validation.

**Excluded**
- `trend_pct` — excluded because it directly describes the trend signal used for the proxy and could create leakage.
- Identifier fields are not used as predictive features because they identify groups/items rather than describe page performance.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify the fields used in the data contract

features = [
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "content_age_days"
]

label_proxy = "trend_direction"

context = [
    "content_id",
    "client_id"
]

excluded = [
    "trend_pct"
]

print("FEATURES:")
for col in features:
    print("-", col, "->", "available" if col in df.columns else "MISSING")

print("\nLABEL / PROXY:")
print("-", label_proxy, "->", "available" if label_proxy in df.columns else "MISSING")

print("\nCONTEXT:")
for col in context:
    print("-", col, "->", "available" if col in df.columns else "MISSING")

print("\nEXCLUDED:")
for col in excluded:
    print("-", col, "->", "present" if col in df.columns else "not present")

assert all(col in df.columns for col in features)
assert label_proxy in df.columns
assert all(col in df.columns for col in context)

print("\nField contract check passed.")

FEATURES:
- impressions_90d -> available
- sessions_90d -> available
- ctr -> available
- avg_position -> available
- content_age_days -> available

LABEL / PROXY:
- trend_direction -> available

CONTEXT:
- content_id -> available
- client_id -> available

EXCLUDED:
- trend_pct -> present

Field contract check passed.


In [5]:
# Check the actual March 2026 warehouse columns

schema = con.execute(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{MAR}')
""").fetchdf()

print(schema[["column_name", "column_type"]].to_string(index=False))

             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              ai_copilot      BIGINT
 

In [6]:
# ML-04: five-feature contract for the March 2026 slice

feature_contract = pd.DataFrame([
    {
        "feature": "gsc_impressions",
        "rationale": "Measures observed search visibility for the page.",
        "availability?": "gsc_data_available IS TRUE"
    },
    {
        "feature": "gsc_clicks",
        "rationale": "Measures observed clicks from search results.",
        "availability?": "gsc_data_available IS TRUE"
    },
    {
        "feature": "gsc_avg_position",
        "rationale": "Measures the observed average search position.",
        "availability?": "gsc_data_available IS TRUE"
    },
    {
        "feature": "ga4_sessions",
        "rationale": "Measures observed user sessions from GA4.",
        "availability?": "ga4_data_available IS TRUE"
    },
    {
        "feature": "content_created_date",
        "rationale": "Provides content maturity information without using a future outcome.",
        "availability?": "Available in dim_content"
    }
])

feature_contract

,feature,rationale,availability?
0,gsc_impressions,Measures observed search visibility for the page.,gsc_data_available IS TRUE
1,gsc_clicks,Measures observed clicks from search results.,gsc_data_available IS TRUE
2,gsc_avg_position,Measures the observed average search position.,gsc_data_available IS TRUE
3,ga4_sessions,Measures observed user sessions from GA4.,ga4_data_available IS TRUE
4,content_created_date,Provides content maturity information without ...,Available in dim_content


In [7]:
# Deliberate leakage trap:
# A feature created directly from the label can make model performance look perfect.

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

leakage_demo = con.execute(f"""
    SELECT
        gsc_impressions,
        gsc_avg_position,
        gsc_clicks
    FROM read_parquet('{MAR}')
    WHERE gsc_data_available IS TRUE
    LIMIT 10000
""").fetchdf()

# Temporary demonstration label
leakage_demo["temporary_label"] = (
    leakage_demo["gsc_clicks"] < 10
).astype(int)

# Deliberately leaked feature: directly copies the label
leakage_demo["leaked_feature"] = leakage_demo["temporary_label"]

X = leakage_demo[["leaked_feature"]]
y = leakage_demo["temporary_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = DecisionTreeClassifier(max_depth=1, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
score = accuracy_score(y_test, predictions)

print("Temporary label distribution:")
print(y.value_counts())

print("\nLeaked feature equals label:",
      (leakage_demo["leaked_feature"] == y).all())

print("Accuracy with leaked feature:", round(score, 3))

assert (leakage_demo["leaked_feature"] == y).all()
assert score == 1.0

print("\nLeakage trap confirmed: the leaked feature gives perfect accuracy.")
print("The leaked feature must be removed from the final feature set.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Temporary label distribution:
temporary_label
1    9975
0      25
Name: count, dtype: int64

Leaked feature equals label: True
Accuracy with leaked feature: 1.0

Leakage trap confirmed: the leaked feature gives perfect accuracy.
The leaked feature must be removed from the final feature set.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1: verify the March 2026 date window

q1 = con.execute(f"""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(*) AS row_count
    FROM read_parquet('{MAR}')
""").fetchdf()

print(q1)

assert str(q1.loc[0, "min_date"])[:7] == "2026-03"
assert str(q1.loc[0, "max_date"])[:7] == "2026-03"

print("\nMarch 2026 window check passed.")

    min_date   max_date  row_count
0 2026-03-01 2026-03-31    9841378

March 2026 window check passed.


In [9]:
# Query 2: verify GA4 availability using IS TRUE

q2 = con.execute(f"""
    SELECT
        ga4_data_available,
        COUNT(*) AS row_count
    FROM read_parquet('{MAR}')
    WHERE ga4_data_available IS TRUE
    GROUP BY ga4_data_available
    ORDER BY ga4_data_available
""").fetchdf()

print(q2)

assert len(q2) == 1
assert q2.loc[0, "ga4_data_available"] is True or bool(q2.loc[0, "ga4_data_available"])

print("\nAvailability check passed: GA4 data is filtered with IS TRUE.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   ga4_data_available  row_count
0                True     413966

Availability check passed: GA4 data is filtered with IS TRUE.


In [10]:
# Query 3: verify the warehouse grain for March 2026

q3 = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_content,
        COUNT(DISTINCT report_date) AS unique_dates
    FROM read_parquet('{MAR}')
""").fetchdf()

print(q3)

print("\nGrain:")
print("One row = one client × content × report date.")

   total_rows  unique_clients  unique_content  unique_dates
0     9841378              55          331437            31

Grain:
One row = one client × content × report date.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset has several limitations that affect the analysis:

- The history is **not balanced** across all clients, so different clients can have different periods of available data.
- GA4 fields are only reliable when `ga4_data_available IS TRUE`.
- The March 2026 slice is used for development, while the final month should be treated as a sealed test period.
- The data provides **observed and directional signals** for decision support. It cannot prove that a content change caused a performance change.
- The available data does not guarantee that a page will recover after a refresh or predict how a search engine will respond.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify the main availability limitation

availability_summary = con.execute(f"""
    SELECT
        gsc_data_available,
        ga4_data_available,
        COUNT(*) AS row_count
    FROM read_parquet('{MAR}')
    GROUP BY gsc_data_available, ga4_data_available
    ORDER BY gsc_data_available, ga4_data_available
""").fetchdf()

availability_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_data_available,ga4_data_available,row_count
0,False,False,4690323
1,False,True,49619
2,False,<NA>,1490375
3,True,False,1718348
4,True,True,364347
5,True,<NA>,1528366


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.